# Documentacion de lo realizado
**Objetivo**: Obtener un conjunto de datos limpio, geolocalizado y representativo del tráfico de la M-30 para el entrenamiento de modelos predictivos de series temporales.

1. Pasos Realizados (Fases 1 al 4)
- Carga y Fusión de Datos: Se han importado los metadatos de los sensores (ubicaciones_m30.csv) y las series temporales históricas (dataset_completo.parquet). Ambos conjuntos se cruzaron mediante un inner join utilizando el identificador único del sensor (id).
- Filtrado Temporal: Se descartaron los datos anteriores al 1 de enero de 2024.
- Control de Calidad: Se evaluó la completitud del dataset, confirmando un 0% de valores nulos en las variables predictoras tras el cruce, lo que denota una alta fiabilidad de las lecturas horarias pre-procesadas.
- Ingeniería de Características: Se calculó la media y, fundamentalmente, la varianza de la intensidad para cada sensor.
- Selección Espacial: Se identificaron los 4 sensores definitivos que conformarán el objeto de estudio.

2. Justificación Metodológica de las Decisiones
- Uso de datos pre-agregados a nivel horario: Trabajar con frecuencias de 1 hora (en lugar de los 15 minutos originales) actúa como un filtro natural que elimina el ruido de alta frecuencia (pequeños atascos, semáforos, incidentes efímeros). Esto optimiza el coste computacional y permite a los modelos (estadísticos y Deep Learning) centrarse en capturar los patrones macro-estacionales (horas punta diurnas, valles nocturnos).
- Acotación Temporal (Desde 2024): Para garantizar que los algoritmos predictivos aprendan de la dinámica vial actual, se ha restringido el historial al año 2024 en adelante, evitando que patrones obsoletos introduzcan sesgos en los pesos de las redes neuronales o en los coeficientes autorregresivos.
- Metodología de Selección de Sensores (Heurística Espacial + Varianza): Para cumplir con el objetivo de obtener 4 nodos representativos (Norte, Sur, Este y Oeste), se descartó el uso de algoritmos de clustering no supervisado (como K-Means), ya que estos agrupan por pura similitud matemática y no garantizan una distribución geográfica equilibrada. En su lugar, se implementó un algoritmo heurístico: 
- Se utilizaron los percentiles 20 y 80 de las coordenadas (latitud y longitud) para aislar los cuadrantes geográficos extremos de la M-30.
- Dentro de cada cuadrante, se seleccionó el nodo con la mayor varianza de intensidad histórica. Esto asegura matemáticamente que los modelos se entrenarán con series temporales ricas en fluctuaciones, proporcionando una señal fuerte y clara para poner a prueba la capacidad predictiva de los modelos ARIMA y LSTM.


# Fase 1: Cargue y visualizacion inicial de los datos

In [2]:
import pandas as pd 

# Cargamos el fichero con las ubicaciones de los sensores M30
df_ubicaciones = pd.read_csv('../data/extra/ubicaciones_m30.csv')

# Cargamos fichero parquet con los datos historicos con los sensosres M30
df_trafico = pd.read_parquet('../data/processed/dataset_completo.parquet', engine='pyarrow')

# 3. Inspección inicial: Identificamos las dimensiones de nuestros datos
print("--- DIMENSIONES DE LOS DATOS ---")
print(f"Catálogo de Sensores (Ubicaciones): {df_ubicaciones.shape[0]} filas y {df_ubicaciones.shape[1]} columnas.")
print(f"Registros Históricos (Tráfico): {df_trafico.shape[0]:,} filas y {df_trafico.shape[1]} columnas.\n".replace(',', '.'))

# 4. Ver las primeras líneas de cada DataFrame
print("--- MUESTRA DE UBICACIONES ---")
# 'display()' es una función especial de Jupyter que formatea la tabla de forma visual
display(df_ubicaciones.head(3)) 

print("\n--- MUESTRA DE TRÁFICO ---")
display(df_trafico.head(3))

--- DIMENSIONES DE LOS DATOS ---
Catálogo de Sensores (Ubicaciones): 353 filas y 11 columnas.
Registros Históricos (Tráfico): 9.624.596 filas y 6 columnas.

--- MUESTRA DE UBICACIONES ---


,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud,tipo_elem_aux,tipo_elem_m30
0,M30,3.0,6668,PM10768,PM10768,"444001,8459","4474331,118",-3.660063,40.417722,M30,1
1,M30,3.0,6669,PM10766,PM10766,"443977,812","4474303,082",-3.660344,40.417468,M30,1
2,M30,2.0,6688,PM11303,PM11303,"441272,9676","4470633,619",-3.691886,40.384225,M30,1



--- MUESTRA DE TRÁFICO ---


,id,fecha,intensidad,ocupacion,carga,vmed
0,1012,2022-01-01 00:00:00,36.00,1.00,0.0,11.00
1,1012,2022-01-01 01:00:00,509.25,4.25,0.0,53.25
2,1012,2022-01-01 02:00:00,441.00,3.75,0.0,51.00


# Fase 2: Preparacion de los datos

In [3]:
print("Iniciando Data Preparation Avanzada...")

# 1. Asegurarnos de que Pandas entiende la fecha
df_trafico['fecha'] = pd.to_datetime(df_trafico['fecha'])

# 2. FILTRADO TEMPORAL (Desde 2024)
fecha_corte = pd.to_datetime('2024-01-01 00:00:00')
df_trafico_2024 = df_trafico[df_trafico['fecha'] >= fecha_corte].copy()
total_inicial = df_trafico_2024.shape[0]
print(f"-> Registros filtrados (>= 2024): {total_inicial:,}".replace(',', '.'))

# 3. TRAZABILIDAD DE NULOS
columnas_clave = ['intensidad', 'ocupacion', 'carga', 'vmed']
nulos_por_columna = df_trafico_2024[columnas_clave].isnull().sum()
filas_con_nulos = df_trafico_2024[columnas_clave].isnull().any(axis=1).sum()
porcentaje_nulos = (filas_con_nulos / total_inicial) * 100

print("\n--- REPORTE DE CALIDAD DE DATOS (NULOS) ---")
print(nulos_por_columna)
print(f"Total de filas con al menos un nulo: {filas_con_nulos:,} ({porcentaje_nulos:.2f}%)".replace(',', '.'))

# 4. TRATAMIENTO DE NULOS EN SERIES TEMPORALES
# En lugar de dropna(), usamos ffill() (Forward Fill): rellenenamos con el ultimo valor valido 
df_trafico_2024 = df_trafico_2024.sort_values(by=['id', 'fecha'])
df_trafico_2024[columnas_clave] = df_trafico_2024.groupby('id')[columnas_clave].ffill()

# 5. UNIÓN DE TABLAS (Merge)
df_maestro = pd.merge(
    left=df_trafico_2024, 
    right=df_ubicaciones[['id', 'latitud', 'longitud', 'nombre']], 
    on='id', 
    how='inner'
)
print(f"\n-> Dimensiones del DataFrame Maestro tras imputación y cruce: {df_maestro.shape[0]:,} filas.".replace(',', '.'))

display(df_maestro.head(3))

Iniciando Data Preparation Avanzada...
-> Registros filtrados (>= 2024): 5.656.138

--- REPORTE DE CALIDAD DE DATOS (NULOS) ---
intensidad    0
ocupacion     0
carga         0
vmed          0
dtype: int64
Total de filas con al menos un nulo: 0 (0.00%)

-> Dimensiones del DataFrame Maestro tras imputación y cruce: 5.656.138 filas.


,id,fecha,intensidad,ocupacion,carga,vmed,latitud,longitud,nombre
0,1012,2024-01-01 00:00:00,124.0,0.666667,0.0,47.666667,40.419861,-3.722097,18RA66PM01
1,1012,2024-01-01 01:00:00,843.0,7.500000,0.0,50.250000,40.419861,-3.722097,18RA66PM01
2,1012,2024-01-01 02:00:00,627.0,5.250000,0.0,50.500000,40.419861,-3.722097,18RA66PM01


# Fase 3: Selección Espacial de 4 sensores representativos (Norte, Sur, Este y Oeste).

In [4]:
print("Agregando datos históricos y calculando métricas de tráfico...")

# 1. Agrupamos por sensor (id) para calcular la varianza y mantener su ubicación
# Usamos first para tomar el primer valor de l agrupacion al que se asocie ese registro
df_kpis = df_maestro.groupby('id').agg(
    intensidad_media=('intensidad', 'mean'),
    intensidad_varianza=('intensidad', 'var'),
    latitud=('latitud', 'first'),
    longitud=('longitud', 'first'),
    nombre=('nombre', 'first')
).reset_index()

# 2. Definimos los umbrales geográficos (percentiles 20 y 80) para aislar los extremos
lat_80 = df_kpis['latitud'].quantile(0.80) # Límite Norte
lat_20 = df_kpis['latitud'].quantile(0.20) # Límite Sur
lon_80 = df_kpis['longitud'].quantile(0.80) # Límite Este
lon_20 = df_kpis['longitud'].quantile(0.20) # Límite Oeste

# 3. Creamos subconjuntos para cada zona cardianal
sensores_norte = df_kpis[df_kpis['latitud'] >= lat_80]
sensores_sur   = df_kpis[df_kpis['latitud'] <= lat_20]
sensores_este  = df_kpis[df_kpis['longitud'] >= lon_80]
sensores_oeste = df_kpis[df_kpis['longitud'] <= lon_20]

# 4. Seleccionamos el sensor con mayor varianza de tráfico en cada zona
seleccion = []

def obtener_mejor_sensor(df_zona, etiqueta):
    # Obtenemos el índice del sensor con la mayor varianza
    idx_max_var = df_zona['intensidad_varianza'].idxmax()
    mejor_sensor = df_zona.loc[idx_max_var].copy()
    mejor_sensor['zona_cardinal'] = etiqueta
    return mejor_sensor

seleccion.append(obtener_mejor_sensor(sensores_norte, 'Norte'))
seleccion.append(obtener_mejor_sensor(sensores_sur, 'Sur'))
seleccion.append(obtener_mejor_sensor(sensores_este, 'Este'))
seleccion.append(obtener_mejor_sensor(sensores_oeste, 'Oeste'))

# 5. Consolidamos el DataFrame final con los 4 sensores estratégicos
df_4_sensores = pd.DataFrame(seleccion)
# Reordenamos columnas para que sea más legible
df_4_sensores = df_4_sensores[['zona_cardinal', 'id', 'nombre', 'intensidad_media', 'intensidad_varianza', 'latitud', 'longitud']]

print("\n--- SELECCIÓN FINAL: 4 SENSORES ESTRATÉGICOS ---")
display(df_4_sensores)

Agregando datos históricos y calculando métricas de tráfico...

--- SELECCIÓN FINAL: 4 SENSORES ESTRATÉGICOS ---


,zona_cardinal,id,nombre,intensidad_media,intensidad_varianza,latitud,longitud
189,Norte,6775,PM23111,2469.092599,2.706991e+06,40.483643,-3.688500
99,Sur,6679,PM11071,3865.055122,4.324614e+06,40.393664,-3.674823
54,Este,3819,PM10502,4148.516428,5.565500e+06,40.440890,-3.659560
6,Oeste,1018,18XC82PM01,2824.697446,3.413879e+06,40.421816,-3.724761


# Fase 4: Visualizacion en el mapa de Madrid

In [6]:
import plotly.express as px

print("Generando mapa interactivo con estilo Carto-Positron en el navegador...")

# Usamos px.scatter_map c
fig = px.scatter_map(
    df_4_sensores, 
    lat="latitud", 
    lon="longitud", 
    hover_name="zona_cardinal", 
    # Datos que aparecerán al pasar el cursor
    hover_data={
        "id": True, 
        "nombre": True, 
        "intensidad_media": ":.2f",
        "intensidad_varianza": ":.2e",
        "latitud": False, 
        "longitud": False
    },
    color="zona_cardinal",
    # Mapeo de colores: asignamos colores claros y distintivos
    color_discrete_map={
        "Norte": "#FF0000",   # Rojo
        "Sur": "#FF8C00",     # Naranja
        "Este": "#0000FF",    # Azul
        "Oeste": "#008000"    # Verde
    },
    zoom=11, 
    map_style="carto-positron", # Estilo limpio y profesional solicitado
    title="Sensores Estratégicos Seleccionados para el TFM (M30)"
)

# Ajuste del tamaño de los puntos y opacidad para mejorar la visibilidad
fig.update_traces(marker=dict(size=12, opacity=0.8))

# Ajuste de márgenes
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})

# Abrir en el navegador para asegurar la visualización
fig.show(renderer="browser")

Generando mapa interactivo con estilo Carto-Positron en el navegador...
